# 02 -- Context Window Chunking Comparison Demo

Demonstrates the Chapter 3 point concretely, entirely offline: given one synthetic **long document**
(a fabricated multi-year prior-case-note history), this notebook builds two workflows over it --

1. **Small-context-window workflow**: the document doesn't fit in one pass, so it's chunked,
   each chunk is summarized independently (a naive, deterministic mock summarizer -- no real LLM),
   and a final narrative is synthesized *only* from the chunk summaries. This is the
   chunk-and-summarize workaround Chapter 3 describes.
2. **Large-context-window workflow**: the document fits in one pass, so a mock "model" reads the
   whole thing at once and can draw a cross-chunk connection the chunked workflow structurally
   cannot see.

The goal is to make the "chunking artifact" failure mode from Chapter 3 -- a fact that only becomes
meaningful when two passages are read together, lost because they were summarized in isolation --
visible and measurable, not just asserted in prose. Chapter 3 now also applies this same mechanism to
a related failure mode in this course's multi-agent system: information lost when an upstream agent's
full output is compressed before being handed off to a downstream agent, rather than lost between
chunks of one document -- the same underlying pattern, at a different seam.

In [1]:
import textwrap
import pandas as pd

pd.set_option("display.max_colwidth", 100)
print("Environment ready. Offline, deterministic, no network calls.")

Environment ready. Offline, deterministic, no network calls.


## 1. The synthetic long document

A fabricated multi-year prior-case-note history for one customer, built from several dated
investigator entries. The two entries that matter for this demo are entry 1 (2021) and entry 5
(2024) -- read together, they reveal a pattern (the same unverified claim reused three years later);
read in isolation, neither one looks remarkable on its own.

In [2]:
CASE_NOTE_ENTRIES = [
    ("2021-03-11", "Customer flagged for irregular deposit pattern. Explained as inheritance-"
     "related funds from a relative's estate. Investigator requested and received estate "
     "documentation confirming the claim. Case closed, false positive."),
    ("2021-09-02", "Routine account review. No unusual activity. Customer's stated occupation "
     "(freelance consultant) consistent with observed transaction volume."),
    ("2022-05-19", "Minor alert for a single transaction slightly above the customer's typical "
     "monthly volume. Reviewed and closed same day, within normal seasonal variation for "
     "consulting income."),
    ("2023-01-30", "Customer requested a credit limit increase on a linked product. No AML "
     "relevance; routed to the credit team."),
    ("2024-02-14", "New alert: irregular deposit pattern, again explained by the customer as "
     "inheritance-related funds. Investigator accepted the explanation without re-requesting "
     "verification documentation this time, citing the 2021 case as precedent. Case closed."),
]

full_document = "\n\n".join(f"[{date}] {text}" for date, text in CASE_NOTE_ENTRIES)
print(f"Synthetic document length: {len(full_document)} characters, "
      f"{len(CASE_NOTE_ENTRIES)} dated entries.")
print()
print(textwrap.fill(full_document, width=100))

Synthetic document length: 963 characters, 5 dated entries.

[2021-03-11] Customer flagged for irregular deposit pattern. Explained as inheritance-related funds
from a relative's estate. Investigator requested and received estate documentation confirming the
claim. Case closed, false positive.  [2021-09-02] Routine account review. No unusual activity.
Customer's stated occupation (freelance consultant) consistent with observed transaction volume.
[2022-05-19] Minor alert for a single transaction slightly above the customer's typical monthly
volume. Reviewed and closed same day, within normal seasonal variation for consulting income.
[2023-01-30] Customer requested a credit limit increase on a linked product. No AML relevance;
routed to the credit team.  [2024-02-14] New alert: irregular deposit pattern, again explained by
the customer as inheritance-related funds. Investigator accepted the explanation without re-
requesting verification documentation this time, citing the 2021 case as 

## 2. Small-context-window workflow: chunk, summarize, then synthesize

Simulate a small context window by imposing a strict per-chunk character budget, well below the full
document's length, forcing multiple chunks. Each chunk is summarized **independently** -- the
summarizer never sees any other chunk -- which is exactly what makes the cross-chunk pattern
structurally invisible to it.

In [3]:
SMALL_WINDOW_CHUNK_BUDGET = 260  # characters per chunk -- deliberately small to force chunking


def chunk_document(entries, budget):
    chunks, current = [], []
    current_len = 0
    for date, text in entries:
        entry_text = f"[{date}] {text}"
        if current and current_len + len(entry_text) > budget:
            chunks.append(current)
            current, current_len = [], 0
        current.append((date, text))
        current_len += len(entry_text)
    if current:
        chunks.append(current)
    return chunks


chunks = chunk_document(CASE_NOTE_ENTRIES, SMALL_WINDOW_CHUNK_BUDGET)
print(f"Document split into {len(chunks)} chunks under a {SMALL_WINDOW_CHUNK_BUDGET}-char budget.")
for i, chunk in enumerate(chunks, start=1):
    dates = ", ".join(d for d, _ in chunk)
    print(f"  Chunk {i}: entries dated {dates}")

Document split into 5 chunks under a 260-char budget.
  Chunk 1: entries dated 2021-03-11
  Chunk 2: entries dated 2021-09-02
  Chunk 3: entries dated 2022-05-19
  Chunk 4: entries dated 2023-01-30
  Chunk 5: entries dated 2024-02-14


In [4]:
def naive_summarize_chunk(chunk):
    # A deterministic, dependency-free mock "summarizer": keeps the date and the first clause
    # of each entry in the chunk. No real LLM call -- stands in for one, and (crucially) has
    # zero visibility into any other chunk.
    lines = []
    for date, text in chunk:
        first_clause = text.split(".")[0]
        lines.append(f"{date}: {first_clause}.")
    return " ".join(lines)


chunk_summaries = [naive_summarize_chunk(c) for c in chunks]
for i, s in enumerate(chunk_summaries, start=1):
    print(f"Summary of chunk {i}: {s}")

Summary of chunk 1: 2021-03-11: Customer flagged for irregular deposit pattern.
Summary of chunk 2: 2021-09-02: Routine account review.
Summary of chunk 3: 2022-05-19: Minor alert for a single transaction slightly above the customer's typical monthly volume.
Summary of chunk 4: 2023-01-30: Customer requested a credit limit increase on a linked product.
Summary of chunk 5: 2024-02-14: New alert: irregular deposit pattern, again explained by the customer as inheritance-related funds.


In [5]:
def synthesize_from_summaries(summaries):
    # The final narrative-generation step for the small-context-window workflow: it only ever
    # sees the independently-produced chunk summaries, never the original full document.
    return "Case narrative (chunked workflow): " + " ".join(summaries)


small_window_narrative = synthesize_from_summaries(chunk_summaries)
print(textwrap.fill(small_window_narrative, width=100))

Case narrative (chunked workflow): 2021-03-11: Customer flagged for irregular deposit pattern.
2021-09-02: Routine account review. 2022-05-19: Minor alert for a single transaction slightly above
the customer's typical monthly volume. 2023-01-30: Customer requested a credit limit increase on a
linked product. 2024-02-14: New alert: irregular deposit pattern, again explained by the customer as
inheritance-related funds.


## 3. Large-context-window workflow: process the whole document at once

No chunking, no independent per-chunk summarization step -- a mock "model" reads every entry in one
pass and can explicitly flag a fact that spans two entries, because nothing about this workflow ever
loses sight of the full document at once.

In [6]:
def synthesize_from_full_document(entries):
    # Simulates a large-context-window model: it sees every entry simultaneously and can
    # cross-reference across them -- something the chunked workflow above structurally cannot do,
    # because no single summarization step in that workflow ever saw more than one chunk.
    inheritance_claims = [
        (date, text) for date, text in entries if "inheritance" in text.lower()
    ]
    lines = [f"{date}: {text.split('.')[0]}." for date, text in entries]
    narrative = "Case narrative (whole-document workflow): " + " ".join(lines)

    if len(inheritance_claims) >= 2:
        first_date, first_text = inheritance_claims[0]
        second_date, second_text = inheritance_claims[-1]
        verified_first = "verification" in first_text.lower() or "documentation" in first_text.lower()
        verified_second = ("documentation" not in second_text.lower()
                            or "without re-requesting" in second_text.lower())
        if verified_first and verified_second:
            narrative += (
                f" CROSS-REFERENCE FLAG: customer cited the same inheritance-related explanation "
                f"on {first_date} (verified with documentation) and again on {second_date} "
                f"(verification not re-requested this time, citing the earlier case as precedent) "
                f"-- a reused, only-partially-verified claim pattern across the full case history."
            )
    return narrative


large_window_narrative = synthesize_from_full_document(CASE_NOTE_ENTRIES)
print(textwrap.fill(large_window_narrative, width=100))

Case narrative (whole-document workflow): 2021-03-11: Customer flagged for irregular deposit
pattern. 2021-09-02: Routine account review. 2022-05-19: Minor alert for a single transaction
slightly above the customer's typical monthly volume. 2023-01-30: Customer requested a credit limit
increase on a linked product. 2024-02-14: New alert: irregular deposit pattern, again explained by
the customer as inheritance-related funds. CROSS-REFERENCE FLAG: customer cited the same
inheritance-related explanation on 2021-03-11 (verified with documentation) and again on 2024-02-14
(verification not re-requested this time, citing the earlier case as precedent) -- a reused, only-
partially-verified claim pattern across the full case history.


## 4. Compare output coherence on the toy synthetic task

The specific thing worth measuring: does the final narrative surface the cross-reference pattern
between the 2021 and 2024 entries? This is a simple, checkable proxy for the "chunking artifact"
information-loss failure mode chapter 3 describes in prose -- here it's a concrete, testable
assertion, not just an assertion in text.

In [7]:
def contains_cross_reference_pattern(narrative):
    lowered = narrative.lower()
    return "cross-reference" in lowered or (
        "2021" in narrative and "2024" in narrative and "precedent" in lowered
    )


comparison = pd.DataFrame(
    [
        {
            "workflow": "small-context-window (chunk + summarize)",
            "num_chunks": len(chunks),
            "surfaced_cross_reference": contains_cross_reference_pattern(small_window_narrative),
            "narrative_length_chars": len(small_window_narrative),
        },
        {
            "workflow": "large-context-window (whole document)",
            "num_chunks": 1,
            "surfaced_cross_reference": contains_cross_reference_pattern(large_window_narrative),
            "narrative_length_chars": len(large_window_narrative),
        },
    ]
)
comparison

,workflow,num_chunks,surfaced_cross_reference,narrative_length_chars
0,small-context-window (chunk + summarize),5,False,421
1,large-context-window (whole document),1,True,735


In [8]:
assert comparison.loc[
    comparison["workflow"].str.startswith("small"), "surfaced_cross_reference"
].iloc[0] == False, "Expected the chunked workflow to miss the cross-chunk pattern by construction."
assert comparison.loc[
    comparison["workflow"].str.startswith("large"), "surfaced_cross_reference"
].iloc[0] == True, "Expected the whole-document workflow to surface the cross-chunk pattern."
print("Confirmed: the chunked workflow misses the cross-reference pattern; the whole-document "
      "workflow catches it. This is the concrete, measurable version of chapter 3's chunking-"
      "artifact argument.")

Confirmed: the chunked workflow misses the cross-reference pattern; the whole-document workflow catches it. This is the concrete, measurable version of chapter 3's chunking-artifact argument.


## 5. The nuance chapter 3 insists on: bigger window alone isn't sufficient

To make the "necessary but not sufficient" point from chapter 3 concrete too: a large context window
that *fits* the whole document is not automatically a large context window that *attends well* to
every part of it. Simulate a degraded whole-document workflow that has room for everything but
exhibits the "lost in the middle" effect -- it reliably attends to the first and last entries but
loses recall on entries buried in the middle, which is a different, and independently checkable,
failure mode from the chunking-artifact one above.

In [9]:
def synthesize_with_middle_degradation(entries):
    # Has room for the whole document (no chunking), but simulates positional degradation:
    # entries strictly in the middle of the document are dropped from the synthesized output,
    # even though nothing forced them out for space reasons -- illustrating that fitting
    # everything in the window doesn't guarantee the model actually used all of it well.
    n = len(entries)
    middle_lo, middle_hi = n // 3, n - n // 3
    kept = [e for i, e in enumerate(entries) if not (middle_lo <= i < middle_hi) or i in (0, n - 1)]
    lines = [f"{date}: {text.split('.')[0]}." for date, text in kept]
    return "Case narrative (whole-document, WITH positional degradation): " + " ".join(lines)


degraded_narrative = synthesize_with_middle_degradation(CASE_NOTE_ENTRIES)
print(textwrap.fill(degraded_narrative, width=100))
print()
surfaced = contains_cross_reference_pattern(degraded_narrative)
print(f"Cross-reference pattern surfaced despite a large window: {surfaced}")
print("A large context window removes the CHUNKING failure mode; it does not, on its own, "
      "guarantee the model actually attended well to everything inside that window -- exactly "
      "the distinction chapter 3 insists on measuring empirically rather than assuming.")

Case narrative (whole-document, WITH positional degradation): 2021-03-11: Customer flagged for
irregular deposit pattern. 2024-02-14: New alert: irregular deposit pattern, again explained by the
customer as inheritance-related funds.

Cross-reference pattern surfaced despite a large window: False
A large context window removes the CHUNKING failure mode; it does not, on its own, guarantee the model actually attended well to everything inside that window -- exactly the distinction chapter 3 insists on measuring empirically rather than assuming.


## Recap

Two independently-checkable results came out of this notebook: chunk-and-summarize measurably lost a
cross-document pattern that a whole-document pass caught (section 4), and a large-but-degraded
whole-document workflow shows that "the document fits" and "the model used the document well" are
two different, separately-testable claims (section 5). That's the empirical shape behind chapter 3's
argument -- a larger context window is a real, mechanical lever against one specific failure mode
(lossy chunking), and it is not, on its own, a substitute for actually measuring output quality.